# Layer-restricted NTK: eigenvalues & eigenvectors up to layer $l$

The NTK is an **exact sum over parameters**, so it splits exactly by layer:

$$\Theta(x,x') = \sum_{j=1}^{L} \Theta_j(x,x'),\qquad \Theta_j(x,x') = \frac{\partial f(x)}{\partial\theta_j}\cdot\frac{\partial f(x')}{\partial\theta_j},$$

where $\theta_j$ are the parameters of NN layer $j$ and $f$ is the **full** PDF output. This notebook shows how to compute the layer-restricted kernel — the cumulative $\Theta_{\le l}=\sum_{j\le l}\Theta_j$ (params of layers $1..l$) or a single layer's $\Theta_l$ — keeping the PDF output unchanged, so the result plugs straight into the existing eigenvalue / eigenvector / $h$-value machinery.

The feature is a single kwarg, `grad_layers`, on the n3fit model's `grid_values_func`. It rides colibri's existing opaque `kwargs` pass-through (`compute_ntk` → `grid_values_func`), so colibri's generic NTK code is untouched. Inside `grid_values_func` the non-selected layers' parameters are frozen with `stop_gradient`, so `jax.jacfwd` contributes zero columns there. The `ntkpdf.config.layer_kwargs` helper builds the `kwargs` frozenset.

In [ ]:
%matplotlib inline
import numpy as np
import jax.numpy as jnp
from matplotlib import pyplot as plt

# Importing ntkpdf first sets KERAS_BACKEND=jax (via ntkpdf/__init__.py).
from ntkpdf.config import layer_kwargs, NN

from colibri.utils import get_pdf_model
from colibri.ntk.ntkutils import compute_ntk, compute_eigendecomposition, get_parameters_all_epochs
from colibri.ntk.eigenvector import eigenvectors_ensemble_at_epoch
from colibri_n3fit.model import _nn_layer_param_ranges
from validphys.loader import Loader

import logging
logging.basicConfig(level=logging.WARNING)

## Load the fit and inspect the layer structure

`get_pdf_model` returns the per-replica n3fit model colibri builds for each NTK evaluation (the production path). `_nn_layer_param_ranges` reports, for each NN `Dense` layer, its contiguous slice of the flat trainable-parameter vector — the same vector `jax.jacfwd` differentiates. `L` is the number of NN layers.

In [ ]:
FIT = "260527-ac-01-ntk-sgd"
REPLICA = 1            # 1-based replica id

fit_spec = Loader().check_fit(FIT)
replicas_path = fit_spec.path / "fit_replicas"

pdf_model = get_pdf_model(FIT, replica_idx=REPLICA)

# The 'nn' selector (NN) excludes the MSR normalisation layer and neutralises the
# prefactor; its (sub)model's trainable vars are exactly the Dense layers.
submodel = pdf_model._ntk_submodel(("impose_msr",))
ranges = _nn_layer_param_ranges(submodel)
L = len(ranges)
print(f"{L} NN layers; flat-param ranges (start, stop): {ranges}")

## Building the `kwargs`

`layer_kwargs(layers, cumulative=True, base=NN)` layers `grad_layers` onto a base selector (default `NN`). With `cumulative=True` it keeps layers $1..\max(\text{layers})$ (the cumulative $\Theta_{\le l}$); with `cumulative=False` it keeps exactly `layers` (e.g. a single $\Theta_l$).

In [ ]:
print("base NN        :", dict(NN))
print("cumulative <=2 :", dict(layer_kwargs((2,))))
print("individual  2  :", dict(layer_kwargs((2,), cumulative=False)))

## The NTK matrix per layer, and the additivity check

`compute_ntk(pdf_model, params, **kwargs)` is colibri's provider. We pass the per-layer kwargs to get each $\Theta_l$ and check $\sum_l \Theta_l = \Theta_{\text{full}}$ exactly (it must, since the kernel is an exact parameter sum). This holds because the prefactor exponents are non-trainable in production fits, so the trainable parameters are exactly the NN layers.

In [ ]:
param_files = get_parameters_all_epochs(replicas_path, REPLICA)
epoch = sorted(param_files)[len(param_files) // 2]   # a representative epoch
params = jnp.load(param_files[epoch])["params"]

ntk_full, shape = compute_ntk(pdf_model, params, **dict(NN))
ntk_layer = [compute_ntk(pdf_model, params, **dict(layer_kwargs((l,), cumulative=False)))[0]
             for l in range(1, L + 1)]

resid = np.max(np.abs(sum(ntk_layer) - ntk_full))
print(f"epoch {epoch}: NTK shape {ntk_full.shape}")
print(f"max|sum_l Theta_l - Theta_full| = {resid:.2e}  (rel {resid / np.abs(ntk_full).max():.1e})")

## Cumulative eigenvalue spectrum by depth

Eigenvalues of $\Theta_{\le 1}, \Theta_{\le 2}, \dots, \Theta_{\le L}=\Theta$. As layers are added the effective rank grows and the spectrum builds up toward the full kernel.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.6), constrained_layout=True)
for l in range(1, L + 1):
    ntk_cum, _ = compute_ntk(pdf_model, params, **dict(layer_kwargs((l,))))
    ev, _ = compute_eigendecomposition(ntk_cum)
    ax.plot(range(1, len(ev) + 1), np.abs(ev), marker=".", ms=4,
            label=rf"$\Theta_{{\leq {l}}}$")
ax.set_yscale("log")
ax.set_xlabel("rank")
ax.set_ylabel("eigenvalue")
ax.set_title(f"Cumulative NTK spectrum by depth (epoch {epoch})")
ax.legend()
plt.show()

## Layer-restricted eigenvectors

The same kwargs go to colibri's ensemble provider `eigenvectors_ensemble_at_epoch` (the streaming path `h_val_grid` uses). The eigenvector array is `(n_replicas, n_flavour*n_xgrid, n_eigenvectors)` — the **same** output space as the full kernel, so these drop into the existing flavour-space plotting / $h$-value machinery unchanged. Distinct `kwargs` are distinct cache keys, so the two calls don't collide.

In [ ]:
vecs_full = eigenvectors_ensemble_at_epoch(
    fit_spec, replicas_path, epoch,
    replica_index_list=(REPLICA,), kwargs=NN,
)["eigenvectors_data"]

vecs_le2 = eigenvectors_ensemble_at_epoch(
    fit_spec, replicas_path, epoch,
    replica_index_list=(REPLICA,), kwargs=layer_kwargs((2,)),
)["eigenvectors_data"]

print("full   eigenvectors:", vecs_full.shape)
print("<=2    eigenvectors:", vecs_le2.shape)

## Notes

- **Why additivity is exact.** Production fits use `FLAV_INFO_NNPDF40` with `trainable: False` prefactor exponents, so the only trainable parameters are the NN `Dense` layers and $\sum_l \Theta_l$ reproduces the full NN kernel. For a fit with *trainable* prefactor exponents those params form a separate contribution outside `grad_layers=(1..L)`, and the per-layer sum would exclude it.
- **Output space unchanged.** Because $f$ stays the full PDF, every $\Theta_{\le l}$ / $\Theta_l$ lives in the same flavour×$x$ space as the full kernel — directly comparable, and compatible with the data metric $M$, the $h$-values, and the `EvolutionOperator`.
- **Through colibri's high-level providers.** `eigenvalue_grid` / `eigenvalues_ensemble` also accept `kwargs=`; pass `layer_kwargs(...)` and a distinct `name=` (the on-disk cache label) per layer selection to build full epoch-trajectory spectra.